# Workshop: Building Gemma 3 from Scratch
## Notebook 5: Gated MLP (GeGLU)

**Estimated Time: 10 minutes**

In addition to attention, transformers use a Feed-Forward Network (FFN) to process each token. Gemma 3 uses **GeGLU (Gated GELU Universal Linear Unit)**. This is a key difference from Gemma 2, which used SwiGLU.

## Learning Objectives:
1. Understand the GeGLU architecture.
2. Implement the `gate_proj`, `up_proj`, and `down_proj` logic.
3. Compare GELU with standard ReLU and understand why GELU matters.
4. Understand why GeGLU is preferred over SwiGLU in Gemma 3.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import math
import sys

print(f"Python version: {sys.version}")
print(f"PyTorch version: {torch.__version__}")

emb_dim = 768  # Gemma 3 hidden_size
hidden_dim = 2048  # Gemma 3 intermediate_size
sliding_window = 1024

## 1. Activation Functions: GELU

In [ ]:
# Compare GELU vs ReLU
x = torch.linspace(-3, 3, 200)
y_relu = F.relu(x)
y_gelu = F.gelu(x, approximate="tanh")  # tanh approximation for speed

plt.figure(figsize=(8, 4))
plt.plot(x.numpy(), y_relu.numpy(), label="ReLU", linewidth=3)
plt.plot(x.numpy(), y_gelu.numpy(), label="GELU (tanh approx)", linewidth=3)
plt.axhline(y=0, color="k", linestyle="--", alpha=0.3)
plt.axvline(x=0, color="k", linestyle="--", alpha=0.3)
plt.legend(loc="upper left", fontsize=12)
plt.title("ReLU vs GELU: Why Gemma 3 uses GELU")
plt.xlabel("Input")
plt.ylabel("Output")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
print("Key difference:")
print(
    f"  At x=-1: ReLU={F.relu(torch.tensor(-1.0)).item():.3f}, GELU={F.gelu(torch.tensor(-1.0), approximate='tanh').item():.3f}"
)
print(
    f"  At x=0:  ReLU={F.relu(torch.tensor(0.0)).item():.3f}, GELU={F.gelu(torch.tensor(0.0), approximate='tanh').item():.3f}"
)
print(
    f"  At x=1:  ReLU={F.relu(torch.tensor(1.0)).item():.3f}, GELU={F.gelu(torch.tensor(1.0), approximate='tanh').item():.3f}"
)
print()
print("GELU has a small dip in negative territory, giving smoother gradient flow.")

## 2. GeGLU vs SwiGLU: Gemma 3 Architecture Choice

| Variant | Formula | Used in |
|---|---|---|
| **SwiGLU** | `SiLU(gate) 	imes linear(up)` | Llama, Gemma 2 |
| **GeGLU** | `GELU(gate) 	imes linear(up)` | **Gemma 3** |

The computation is:
$$Output = W_{down}(GELU(W_{gate} \cdot x) \odot (W_{up} \cdot x))$$

where $\odot$ is element-wise multiplication.

SwiGLU uses SiLU (Sigmoid-Linear Unit) = `x * sigmoid(x)`. GeGLU replaces SiLU with GELU. The difference is subtle but empirical results in Gemma 3 training showed GELU performs better for their data mix.

In [ ]:
class GatedMLP(nn.Module):
    """
    Gemma 3 GeGLU (GELU-based Gated MLP).

    Architecture: x -> gate_proj -> GELU -> element-wise multiply with up_proj -> down_proj
    """

    def __init__(self, d_in, d_hidden):
        super().__init__()
        # Three projections: gate (uses GELU), up (identity), down (output)
        self.gate_proj = nn.Linear(d_in, d_hidden, bias=False)
        self.up_proj = nn.Linear(d_in, d_hidden, bias=False)
        self.down_proj = nn.Linear(d_hidden, d_in, bias=False)

    def forward(self, x):
        # 1. Compute gate and up projections
        gate = self.gate_proj(x)
        up = self.up_proj(x)

        # 2. Apply GELU activation to gate, then element-wise multiply
        # Gemma 3 uses tanh approximation for GELU (same as SiLU in SwiGLU)
        activated_gate = F.gelu(gate, approximate="tanh")
        intermediate = activated_gate * up

        # 3. Project back to input dimension
        return self.down_proj(intermediate)


mlp = GatedMLP(emb_dim, hidden_dim)
sample_input = torch.randn(1, 10, emb_dim)
output = mlp(sample_input)
print(f"Input shape:  {sample_input.shape}")
print(f"Output shape: {output.shape}")
assert output.shape == sample_input.shape
print("✅ GeGLU MLP works correctly!")

## 3. Comparing GeGLU to a Standard MLP

In [ ]:
def count_parameters(model):
    return sum(p.numel() for p in model.parameters())


gated_params = count_parameters(mlp)
print(f"GeGLU MLP parameters: {gated_params:,}")
print(f"  gate_proj:  emb 	imes hidden = {emb_dim * hidden_dim:,}")
print(f"  up_proj:    emb 	imes hidden = {emb_dim * hidden_dim:,}")
print(f"  down_proj:  hidden 	imes emb   = {hidden_dim * emb_dim:,}")

# Define a standard MLP (just one up + down)
standard_mlp = nn.Sequential(
    nn.Linear(emb_dim, hidden_dim, bias=False),
    nn.Linear(hidden_dim, emb_dim, bias=False),
)
standard_params = count_parameters(standard_mlp)
print(f"\nStandard MLP parameters: {standard_params:,}")
print(
    f"\nGeGLU has {gated_params - standard_params:,} more parameters (+{(gated_params / standard_params - 1) * 100:.0f}%)"
)
print(
    "This extra capacity is why GeGLU needs a larger intermediate_size than hidden_size."
)

## 4. Gating Explained

Gating allows the model to dynamically control information flow for each token:

- `gate_proj` decides **which features** from the input to pass through
- `up_proj` projects to a **higher-dimensional space** for richer computation
- `down_proj` projects back to the original dimension

The gate acts as a **per-token, per-feature filter** -- not a static activation.

## Exercise:
Implement a `SwiGLU` variant of the same MLP to compare. SwiGLU uses `SiLU(gate) 	imes up` instead of `GELU(gate) 	imes up`.

In [ ]:
class SwiGLP(nn.Module):
    """SwiGLU variant used in Llama and Gemma 3."""

    def __init__(self, d_in, d_hidden):
        super().__init__()
        self.gate_proj = nn.Linear(d_in, d_hidden, bias=False)
        self.up_proj = nn.Linear(d_in, d_hidden, bias=False)
        self.down_proj = nn.Linear(d_hidden, d_in, bias=False)

    def forward(self, x):
        gate = self.gate_proj(x)
        up = self.up_proj(x)
        # SiLU = x * sigmoid(x) -- this is the SwiGLU difference
        activated_gate = F.silu(gate)  # Same as F.swish(gate)
        intermediate = activated_gate * up
        return self.down_proj(intermediate)


swiglu = SwiGLP(emb_dim, hidden_dim)
print(f"SwiGLU parameters: {count_parameters(swiglu):,}")
print(f"GeGLU parameters:  {count_parameters(mlp):,}")
print("\nBoth have the same parameter count -- the difference is just GELU vs SiLU.")

## Key Takeaway

Gemma 3 GeGLU uses GELU gating with three projections:
- `gate_proj` → GELU(gate)
- `up_proj` → identity
- `down_proj` → output

This is different from Gemma 2 (SwiGLU) which used SiLU gating. Gemma 3 found GELU works better for their training data.